<a href="https://colab.research.google.com/github/sonky20/sonky/blob/master/5%EC%9D%BC%EC%B0%A8_%EB%8B%A4%EC%A4%91%EB%B6%84%EB%A5%98%EB%AA%A8%EB%8D%B8%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#라이브러리 로딩
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam

#다바이스 준비
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
##학습 함수
def train(dataloader, model, loss_fn, optimizer, device):
  size = len(dataloader.dataset) #전체 데이터 세트의 크기
  num_batches = len(dataloader) #배치 크기
  tr_loss = 0

  model.train() #학습 모드로 설정
  for x, y in dataloader: #배치 단위 로딩
    x, y = x.to(device), y.to(device) #디바이스 지정

    #Feed Forward(오차 순전파)
    pred = model(x)
    loss = loss_fn(pred, y)
    tr_loss += loss

    #Backpropagation(오차 역전파)
    loss.backward() #역전파를 통해 각 파라미터의 손실 기울기 계산
    optimizer.step() #옵티마이저가 모델의 파라미터를 업데이트
    optimizer.zero_grad() #옵티마이저의 기울기값 초기화

  tr_loss /= num_batches
  return tr_loss.item()

##검증 평가 함수
def evaluate(x_val_tensor, y_val_tensor, model, loss_fn, device):
  model.eval() #모델을 평가 모드로 설정
  with torch.no_grad(): #기울기 계산 비활성화
    x, y = x_val_tensor.to(device), y_val_tensor.to(device)
    pred = model(x)
    eval_loss = loss_fn(pred, y) #예측값 pred와 목표값 y사이의 오차 계산
  return eval_loss.item(), pred

##학습 곡선 함수
def dl_learning_curve(tr_loss_list, val_loss_list):
  epochs = list(range(1, len(tr_loss_list)+1))
  plt.plot(epochs, tr_loss_list, label='tran_err', marker='.')
  plt.plot(epochs, val_loss_list, label='val_err', marker='.')
  plt.ylable('Loss')
  plt.xlabel('Epochs')
  plt.legend()
  plt.grid()
  plt.show()






In [4]:
##데이터 로더 선언 함수
def make_DataSet(x_train, x_val, y_train, y_val, batch_size = 32):
  #텐서로 변환
  x_train_tensor = torch.tensor(x_train, dtype=torch.float32)
  y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
  x_val_tensor = torch.tensor(x_val, dtype=torch.float32)
  y_val_tensor = torch.tensor(y_val, dtype=torch.long)

  #TensorDataSet 생성: 텐서 데이터 세트로 합치기
  train_dataset = TensorDataset(x_train_tensor, y_train_tensor)

  #DataLoader 생성
  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
  return train_loader, x_val_tensor, y_val_tensor



In [7]:
path = "https://bit.ly/irisdata_csv"
data = pd.read_csv(path)
data.head()


,Sepal.Length,Sepal.Width,Petal.Length,Petal.Width,Species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [8]:
target = 'Species'
x = data.drop(target, axis = 1)
y = data.loc[:, target]